In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!rm -rf /content/Parking
!git clone https://github.com/hou1020/Parking.git /content/Parking
%cd /content/Parking/parking-lot-mapping-tool
!ls

In [ ]:
pwd

In [ ]:
!pip install -q "pytorch-lightning<2.0.0"
!pip install -q "transformers==4.40.2" datasets roboflow
!pip install -q evaluate rasterio geojson imageio geopandas

# Leeds TIF Batch Inference

逐个处理 `files/leeds_tif` 里的 9 个 GeoTIFF，只输出模型原始停车场 polygon，不做 buildings 和 roads 后处理。

In [ ]:
from pathlib import Path
import shutil
import warnings

import geopandas as gpd
import imageio.v2 as iio
import numpy as np
import pandas as pd
import rasterio
import torch
from PIL import Image
from shapely.ops import unary_union
from torch import nn
from torch.utils.data import DataLoader
from tqdm import tqdm
from transformers import SegformerFeatureExtractor
from huggingface_hub import hf_hub_download

from functions import convert_to_rgb, split_images, find_polygons, pixels_to_coordinates
from inference import SemanticSegmentationDataset, SegformerFinetuner

warnings.filterwarnings("ignore")

In [ ]:
TIF_DIR = Path("files/leeds_tif")
OUTPUT_DIR = Path("output_files/leeds_original")
RGB_PATH = Path("files/large_img.PNG")
MODEL_PATH = Path("files/best_model.ckpt")

BATCH_SIZE = 5
NUM_WORKERS = 0
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Device:", DEVICE)
print("TIF count:", len(sorted(TIF_DIR.glob("*.tif"))))

In [ ]:
# 如果 files/best_model.ckpt 不存在，就从 Hugging Face 下载。
# 本地第一次运行这一步需要联网。
if not MODEL_PATH.exists() or MODEL_PATH.stat().st_size < 1024:
    cached_file = hf_hub_download(
        repo_id="UTEL-UIUC/SegFormer-large-parking",
        filename="best_model.ckpt",
    )
    shutil.copy(cached_file, MODEL_PATH)

print("Model checkpoint:", MODEL_PATH)

In [ ]:
# 加载 SegFormer 模型。这里保留 main.ipynb 的模型结构，但支持 CPU/GPU 自动选择。
feature_extractor = SegformerFeatureExtractor.from_pretrained(
    "nvidia/segformer-b5-finetuned-cityscapes-1024-1024"
)
feature_extractor.do_reduce_labels = False
feature_extractor.size = 512

# 先用一个 tif 初始化 small_images，从而创建 dataset 和 id2label。
first_tif = sorted(TIF_DIR.glob("*.tif"))[0]
convert_to_rgb(first_tif, RGB_PATH)
first_img = iio.imread(RGB_PATH)
split_images(first_img, num=0)

test_dataset = SemanticSegmentationDataset("small_images/", feature_extractor)
test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)

segformer_finetuner = SegformerFinetuner(
    test_dataset.id2label,
    train_dataloader=test_dataloader,
    val_dataloader=test_dataloader,
    test_dataloader=test_dataloader,
    metrics_interval=10,
).load_from_checkpoint(MODEL_PATH, id2label=test_dataset.id2label)

segformer_finetuner.model.to(DEVICE)
segformer_finetuner.model.eval()
print("Model loaded")

In [ ]:
def read_coordinate_grids(tif_path):
    """读取 tif 每个像素中心点的空间坐标。"""
    with rasterio.open(tif_path) as dataset:
        height, width = dataset.height, dataset.width
        rows, cols = np.meshgrid(range(height), range(width), indexing="ij")
        transform = dataset.transform
        xs, ys = rasterio.transform.xy(transform, rows, cols, offset="center")
        xs = np.array(xs).reshape(height, width)
        ys = np.array(ys).reshape(height, width)
        crs = dataset.crs
    return xs, ys, crs


def pad_coordinate_grids(xs, ys, target_height, target_width):
    """让坐标数组尺寸和 512 切片后的影像尺寸一致。"""
    xs = xs[:target_height, :target_width]
    ys = ys[:target_height, :target_width]

    if target_height > xs.shape[0]:
        pad_width = ((0, target_height - xs.shape[0]), (0, 0))
        xs = np.pad(xs, pad_width, mode="edge")
        ys = np.pad(ys, pad_width, mode="edge")

    if target_width > xs.shape[1]:
        pad_width = ((0, 0), (0, target_width - xs.shape[1]))
        xs = np.pad(xs, pad_width, mode="edge")
        ys = np.pad(ys, pad_width, mode="edge")

    return xs, ys


def save_original_geojson(polygons, inner_polygons, output_path, crs):
    """保存原始 polygon。CRS 使用输入 tif 的 CRS。"""
    output_path.parent.mkdir(parents=True, exist_ok=True)

    outer_gdf = gpd.GeoDataFrame({"geometry": polygons}, geometry="geometry", crs=crs)
    if inner_polygons:
        inner_union = unary_union(gpd.GeoSeries(inner_polygons, crs=crs))
        outer_gdf["geometry"] = outer_gdf.geometry.apply(lambda geom: geom.difference(inner_union))

    outer_gdf = outer_gdf[outer_gdf.geometry.notna() & ~outer_gdf.geometry.is_empty]
    outer_gdf.to_file(output_path, driver="GeoJSON")


def predict_current_small_images(tile_count, padded_height, padded_width):
    """对当前 small_images/Images 里的切片做预测，并拼回一张完整 mask。"""
    dataset = SemanticSegmentationDataset("small_images/", feature_extractor)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)

    n_cols = padded_width // 512
    seg_t = np.zeros((padded_height, padded_width), dtype=np.uint8)

    with torch.no_grad():
        for batch_idx, batch in enumerate(tqdm(dataloader, desc="Predicting tiles")):
            images = batch["pixel_values"].to(DEVICE)
            masks = batch["labels"].to(DEVICE)
            outputs = segformer_finetuner.model(images, masks)
            logits = outputs[1]
            logits = nn.functional.interpolate(
                logits,
                size=masks.shape[-2:],
                mode="bilinear",
                align_corners=False,
            )
            predicted_masks = logits.argmax(dim=1).cpu().numpy()

            for i, mask in enumerate(predicted_masks):
                tile_index = batch_idx * BATCH_SIZE + i
                if tile_index >= tile_count:
                    continue

                row = tile_index // n_cols
                col = tile_index % n_cols
                seg_t[row * 512:(row + 1) * 512, col * 512:(col + 1) * 512] = mask

    return seg_t

In [ ]:
def process_tif(tif_path):
    """处理单个 tif，并输出 output_files/leeds_original/<name>_original.geojson。"""
    name = tif_path.stem
    output_path = OUTPUT_DIR / f"{name}_original.geojson"
    print(f"\nProcessing {name}")

    xs, ys, crs = read_coordinate_grids(tif_path)

    convert_to_rgb(tif_path, RGB_PATH)
    img = iio.imread(RGB_PATH)
    tile_count, padded_height, padded_width = split_images(img, num=0)
    xs, ys = pad_coordinate_grids(xs, ys, padded_height, padded_width)

    seg_t = predict_current_small_images(tile_count, padded_height, padded_width)
    polygons, inner_polygons = find_polygons(seg_t)
    polygons_coord = pixels_to_coordinates(polygons, xs, ys)
    inner_coord = pixels_to_coordinates(inner_polygons, xs, ys)

    save_original_geojson(polygons_coord, inner_coord, output_path, crs)
    print(f"Saved: {output_path}")
    return output_path

In [ ]:
tif_paths = sorted(TIF_DIR.glob("*.tif"))
outputs = []

for tif_path in tif_paths:
    outputs.append(process_tif(tif_path))

pd.DataFrame({"output_geojson": [str(path) for path in outputs]})